# Bài 2.1: Biểu diễn Văn bản Tiếng Việt với TF-IDF, Word2Vec & PhoW2V

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vietnamese-nlp-mastery/vietnamese-nlp-mastery/blob/main/02_Classical_and_Embeddings/01_tfidf_word2vec_phow2v.ipynb)

> **Mục tiêu bài học:**
> 1. Xây dựng Baseline mạnh mẽ và siêu tốc cho bài toán Phân loại văn bản tiếng Việt bằng **TF-IDF (N-gram trên từ đã tách) + LinearSVC / LogisticRegression**.
> 2. Huấn luyện mô hình nhúng từ **Word2Vec (Skip-gram / CBOW)** với thư viện `gensim` trên văn bản tiếng Việt.
> 3. Tải và khai thác **PhoW2V** (VinAI Research) — bộ Word Embeddings tiếng Việt chất lượng cao huấn luyện trên 20GB văn bản Wikipedia + Tin tức.

In [1]:
import json
import sys
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

sys.path.insert(0, str(Path("../01_Foundations_Preprocessing").resolve()))
from vn_preprocessing import clean_vietnamese_text

# 1. Đọc dữ liệu mẫu UIT-VSFC
sample_file = Path("../datasets/samples/uit_vsfc_sample.jsonl")
with open(sample_file, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

texts = [clean_vietnamese_text(row["sentence"], lowercase=True) for row in data]
labels = [row["sentiment_label"] for row in data]

# 2. Xây dựng TF-IDF + LogisticRegression Pipeline
clf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", LogisticRegression(C=1.0, random_state=42)),
])
clf_pipeline.fit(texts, labels)

test_comment = "Giảng viên dạy rất dễ hiểu, phòng thực hành hiện đại."
pred = clf_pipeline.predict([clean_vietnamese_text(test_comment, lowercase=True)])[0]
print(f"Câu kiểm thử : {test_comment}")
print(f"Dự đoán nhãn : {pred}")

Câu kiểm thử : Giảng viên dạy rất dễ hiểu, phòng thực hành hiện đại.
Dự đoán nhãn : Positive


## 2. Huấn luyện Word2Vec & Sử dụng Pre-trained `PhoW2V`

**PhoW2V** cung cấp 2 phiên bản:
- **Syllable-level (`word2vec_vi_syllables_100d` / `300d`)**: Không cần tách từ trước.
- **Word-level (`word2vec_vi_words_100d` / `300d`)**: Văn bản đầu vào được tách từ bằng `VnCoreNLP` (`trí_tuệ_nhân_tạo`, `xử_lý_ngôn_ngữ_tự_nhiên`).

```python
from gensim.models import KeyedVectors

# Tải trọng số PhoW2V (Word-level 100d)
# Nguồn: https://github.com/datquocnguyen/PhoW2V
phow2v = KeyedVectors.load_word2vec_format("word2vec_vi_words_100dims.txt", binary=False)

# Tìm các từ đồng nghĩa / gần nghĩa nhất với "giảng_viên"
similar_words = phow2v.most_similar("giảng_viên", topn=5)
for word, score in similar_words:
    print(f"{word:<20}: {score:.4f}")
```